# State of Competitive Programming — cross-engine verification and first charts

This notebook does two jobs, and the first one is the reason it exists.

**It recomputes the observatory aggregates on a third engine.** The published figures were
built by dbt on DuckDB and the corpus was normalised by PySpark locally. Databricks is a
third independent path over the same facts, and three engines agreeing to the row is a much
stronger claim than one engine asserting a number. Where a count is reproduced here, it is
compared to the committed export exactly, not eyeballed.

**It draws the first charts.** Those come second, because a chart of a number nobody checked
is decoration.

### What is verified, and what is not

| Aggregate | Verified here | Why |
|---|---|---|
| `obs_rating_distribution` | yes, exactly | needs only user ratings |
| `obs_country_participation` | yes, exactly | needs ratings and declared country |
| `obs_activity_by_year` | yes, all 17 rows | needs the fact slice |
| `obs_tag_landscape` | no — charted only | needs problem attributes not in the slice |
| `obs_organization_participation` | no — charted only | needs the organisation column, deliberately not uploaded |

### No Codeforces handle is present in this workspace

`fct_submission` keys on the handle and `dim_user` is keyed the same way. Neither was uploaded.
`active_users` and `problems_attempted` are DISTINCT COUNTS, and a distinct count is invariant
under any injective relabelling, so the slice carries dense integer surrogates with no preimage.
A salted hash would have been weaker: the set of Codeforces handles is public, so a hash with a
committed salt is reversible by anyone willing to run it over that list.

### This workspace is disposable

Databricks Free Edition is non-commercial and may delete accounts after prolonged inactivity.
The durable artefacts are this notebook and its outputs, committed in `codronahq/lens`. Nothing
here is on any critical path.


## 1. Configuration and the published figures

The export is 836 rows across five files. That is read with plain `json.load` rather than
`spark.read.json`: it is the comparison target, not the workload, and putting it through a
DataFrame would be theatre. Spark does the 23.6M-row half, which is the part that needs it.


In [ ]:
import json

VOLUME = "/Volumes/workspace/default/codrona"


def load_export(table):
    with open(f"{VOLUME}/{table}.json") as handle:
        payload = json.load(handle)
    return payload


TABLES = [
    "obs_rating_distribution",
    "obs_activity_by_year",
    "obs_tag_landscape",
    "obs_country_participation",
    "obs_organization_participation",
]
export = {table: load_export(table) for table in TABLES}

for table in TABLES:
    print(f"{table:<34} {export[table]['row_count']:>5} rows")
print()
print(export["obs_activity_by_year"]["caveat"])

## 2. The comparison rule

Counts are integers and must match to the row; anything less is not agreement. The rounded
percentage columns get a tolerance, because DuckDB rounds half away from zero while Spark
rounds half up, so the two can differ in the last decimal place on an exact half. Stating
which columns got which treatment matters more than the result — a tolerance applied quietly
to everything would let a real disagreement pass as a rounding artefact.


In [ ]:
def compare(label, computed, published, key, int_columns, float_columns, tolerance=0.011):
    """Compare two lists of dicts row by row. Returns a list of failures."""
    left = {row[key]: row for row in computed}
    right = {row[key]: row for row in published}
    failures = []

    only_computed = sorted(set(left) - set(right), key=str)
    only_published = sorted(set(right) - set(left), key=str)
    for missing in only_computed:
        failures.append(f"{label}: {missing!r} computed but not published")
    for missing in only_published:
        failures.append(f"{label}: {missing!r} published but not computed")

    for name in sorted(set(left) & set(right), key=str):
        for column in int_columns:
            a, b = left[name].get(column), right[name].get(column)
            if a is None and b is None:
                continue
            if a is None or b is None or int(a) != int(b):
                failures.append(f"{label}: {name!r}.{column} computed {a} published {b}")
        for column in float_columns:
            a, b = left[name].get(column), right[name].get(column)
            if a is None and b is None:
                continue
            if a is None or b is None or abs(float(a) - float(b)) > tolerance:
                failures.append(f"{label}: {name!r}.{column} computed {a} published {b}")

    rows = len(set(left) & set(right))
    verdict = "AGREES" if not failures else f"{len(failures)} DISAGREEMENT(S)"
    print(f"{label:<32} {rows:>4} rows compared   {verdict}")
    for failure in failures:
        print(f"    {failure}")
    return failures


results = {}

## 3. The user slice

55,484 rows: rating, declared country, registration year. No handle, no name, no city.


In [ ]:
users = spark.read.parquet(f"{VOLUME}/dim_user_slice.parquet")
users.createOrReplaceTempView("dim_user_slice")
print(f"{users.count():,} users")
users.printSchema()

## 4. `obs_rating_distribution`, recomputed

The band boundaries are Codeforces' own. Rating can be negative — the observed minimum is
-19 — so the lowest band is bounded below by nothing rather than by zero.


In [ ]:
bands = spark.sql("""
    with banded as (
        select
            case
                when rating < 1200 then 'newbie'
                when rating < 1400 then 'pupil'
                when rating < 1600 then 'specialist'
                when rating < 1900 then 'expert'
                when rating < 2100 then 'candidate master'
                when rating < 2300 then 'master'
                when rating < 2400 then 'international master'
                when rating < 2600 then 'grandmaster'
                when rating < 3000 then 'international grandmaster'
                else 'legendary grandmaster'
            end as rating_band,
            rating
        from dim_user_slice
    )
    select
        rating_band,
        count(*) as cohort_users,
        round(100.0 * count(*) / sum(count(*)) over (), 3) as cohort_share_pct,
        min(rating) as min_rating,
        max(rating) as max_rating,
        round(avg(rating), 1) as mean_rating
    from banded
    group by rating_band
""").collect()

results["rating"] = compare(
    "obs_rating_distribution",
    [row.asDict() for row in bands],
    export["obs_rating_distribution"]["rows"],
    key="rating_band",
    int_columns=["cohort_users", "min_rating", "max_rating"],
    float_columns=["cohort_share_pct", "mean_rating"],
)

## 5. `obs_country_participation`, recomputed

Two things this table does deliberately, both reproduced here rather than smoothed over.

The undeclared population is a ROW, not a filter. 36,584 of 55,484 users declare no country,
and declaring correlates with strength — declared users average 1212.5 against 937.0. A country
slice is therefore a sample of the stronger, more engaged fraction who filled in a profile
field, and dropping the undeclared row would turn a 34% sample into an apparent census.

Cells below five users publish no rating statistics. 64 of 158 rows are in that state, and the
smallest holds one user, where a mean equals a max equals that person's exact current rating
beside a country that `user.ratedList` is filterable by.


In [ ]:
countries = spark.sql("""
    with grouped as (
        select
            coalesce(country, '(undeclared)') as country,
            count(*) as cohort_users,
            round(100.0 * count(*) / sum(count(*)) over (), 3) as cohort_share_pct,
            round(avg(rating), 1) as mean_rating,
            percentile(rating, 0.5) as median_rating,
            max(rating) as max_rating,
            count(case when rating >= 1900 then 1 end) as candidate_master_plus
        from dim_user_slice
        group by coalesce(country, '(undeclared)')
    )
    select
        country,
        cohort_users,
        cohort_share_pct,
        case when cohort_users >= 5 then mean_rating end as mean_rating,
        case when cohort_users >= 5 then median_rating end as median_rating,
        case when cohort_users >= 5 then max_rating end as max_rating,
        case when cohort_users >= 5 then candidate_master_plus end as candidate_master_plus
    from grouped
""").collect()

results["country"] = compare(
    "obs_country_participation",
    [row.asDict() for row in countries],
    export["obs_country_participation"]["rows"],
    key="country",
    int_columns=["cohort_users", "max_rating", "candidate_master_plus"],
    float_columns=["cohort_share_pct", "mean_rating", "median_rating"],
)

## 6. The fact slice — 23,607,105 rows

Six columns. The first real check is arithmetic and costs nothing: the published
`obs_activity_by_year` partitions the corpus, so its `submissions` column must sum to the row
count of this file. Nothing had ever compared those two numbers before this notebook.


In [ ]:
facts = spark.read.parquet(f"{VOLUME}/fct_submission_all_slice.parquet")
facts.createOrReplaceTempView("fct_submission_slice")

rows_in_slice = facts.count()
published_total = sum(row["submissions"] for row in export["obs_activity_by_year"]["rows"])
published_person = sum(
    row["person_level_submissions"] for row in export["obs_activity_by_year"]["rows"]
)

print(f"rows in the slice                {rows_in_slice:>12,}")
print(f"published submissions, summed    {published_total:>12,}")
print(f"published person-level, summed   {published_person:>12,}")
print()
print("corpus total agrees:", rows_in_slice == published_total)

## 7. `obs_activity_by_year`, recomputed — all 17 rows

`registered_by_then` and `active_share_pct` are deliberately not compared. The model derives
them through a left join on registration year, which yields NULL for a year in which nobody in
the cohort registered; reproducing that here would be reproducing a join quirk rather than
checking a figure. The seven columns that describe the year's activity are compared exactly.


In [ ]:
activity = spark.sql("""
    select
        submitted_year,
        count(*) as submissions,
        count(case when is_person_level then 1 end) as person_level_submissions,
        count(distinct user_ref) as active_users,
        count(distinct problem_ref) as problems_attempted,
        count(case when is_accepted then 1 end) as accepted,
        round(100.0 * count(case when is_accepted then 1 end) / count(*), 2) as accepted_pct,
        count(case when is_contest then 1 end) as in_contest
    from fct_submission_slice
    group by submitted_year
""").collect()

results["activity"] = compare(
    "obs_activity_by_year",
    [row.asDict() for row in activity],
    export["obs_activity_by_year"]["rows"],
    key="submitted_year",
    int_columns=[
        "submissions",
        "person_level_submissions",
        "active_users",
        "problems_attempted",
        "accepted",
        "in_contest",
    ],
    float_columns=["accepted_pct"],
)

## 8. Verdict


In [ ]:
total_failures = sum(len(failures) for failures in results.values())
print(f"three aggregates recomputed on Databricks, {total_failures} disagreement(s)")
if total_failures:
    raise AssertionError(
        "the published export and this engine disagree - investigate before charting"
    )
print("DuckDB, local PySpark and Databricks agree.")

## 9. The charts

The activity chart carries its own warning in the figure rather than in a caption that a
screenshot would lose. The cohort was collected with `activeOnly=true`, so the final year
reads as fully active because it cannot read otherwise, and the volume curve is the cohort's
registration curve rather than the growth of competitive programming.


In [ ]:
import matplotlib.pyplot as plt

RATING_COLOURS = {
    "newbie": "#808080",
    "pupil": "#008000",
    "specialist": "#03a89e",
    "expert": "#0000ff",
    "candidate master": "#aa00aa",
    "master": "#ff8c00",
    "international master": "#ff8c00",
    "grandmaster": "#e5484d",
    "international grandmaster": "#e5484d",
    "legendary grandmaster": "#e5484d",
}

bands_rows = sorted(export["obs_rating_distribution"]["rows"], key=lambda r: r["band_order"])
figure, axes = plt.subplots(figsize=(11, 5))
axes.bar(
    [row["rating_band"] for row in bands_rows],
    [row["cohort_users"] for row in bands_rows],
    color=[RATING_COLOURS[row["rating_band"]] for row in bands_rows],
)
axes.set_yscale("log")
axes.set_ylabel("cohort users (log scale)")
axes.set_title("Rating distribution — 55,484 collected users, stratified cohort, not a census")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
activity_rows = sorted(export["obs_activity_by_year"]["rows"], key=lambda r: r["submitted_year"])
years = [row["submitted_year"] for row in activity_rows]
volume = [row["submissions"] for row in activity_rows]
partial = [row["is_partial_year"] for row in activity_rows]

figure, axes = plt.subplots(figsize=(11, 5))
axes.bar(
    years,
    volume,
    color=["#c0c0c0" if flag else "#0000ff" for flag in partial],
    hatch=["//" if flag else "" for flag in partial],
)
axes.set_ylabel("submissions")
axes.set_title("Submissions per year — THIS IS NOT A GROWTH CURVE")
axes.annotate(
    "hatched: partial year, truncated by the collection window",
    xy=(0.02, 0.94),
    xycoords="axes fraction",
    fontsize=9,
)
axes.annotate(
    "every user was active in the collection window by construction, so early years\\n"
    "contain only today's-active users who had already registered",
    xy=(0.02, 0.80),
    xycoords="axes fraction",
    fontsize=9,
)
plt.tight_layout()
plt.show()

In [ ]:
declared = [row for row in export["obs_country_participation"]["rows"] if not row["is_undeclared"]]
top = sorted(declared, key=lambda r: r["cohort_users"], reverse=True)[:15]
undeclared = next(
    row for row in export["obs_country_participation"]["rows"] if row["is_undeclared"]
)

figure, axes = plt.subplots(figsize=(11, 5))
axes.barh(
    [row["country"] for row in reversed(top)],
    [row["cohort_users"] for row in reversed(top)],
    color="#03a89e",
)
axes.set_xlabel("cohort users")
axes.set_title("Declared country, top 15 — a sample of those who filled in the field")
axes.annotate(
    f"{undeclared['cohort_users']:,} users declare no country and are not shown here;\\n"
    "declared users average 1212.5 rating against 937.0 undeclared",
    xy=(0.35, 0.10),
    xycoords="axes fraction",
    fontsize=9,
)
plt.tight_layout()
plt.show()

In [ ]:
tags = sorted(
    export["obs_tag_landscape"]["rows"], key=lambda r: r["problems_with_tag"], reverse=True
)[:20]

figure, axes = plt.subplots(figsize=(11, 6))
axes.barh(
    [row["tag"] for row in reversed(tags)],
    [row["problems_with_tag"] for row in reversed(tags)],
    color="#aa00aa",
)
axes.set_xlabel("problems carrying the tag")
axes.set_title("Topic landscape — a problem appears once per tag, so these do not sum to a total")
plt.tight_layout()
plt.show()